In [ ]:
import pandas as pd
import re
import os
from collections import Counter

reviews = pd.read_parquet("data/processed/reviews_clean.parquet")

dis = reviews[reviews["dissent_neg"] & reviews["long_enough"]]
print(len(dis))   

16816


In [ ]:
anchors = {"review", "reviews", "reviewer", "reviewers", "everyone", "everybody",
           "people", "others", "because", "reason", "why", "due", "result"}

counts = Counter()
for text in dis["text_clean"].str.lower():
    words = re.findall(r"[a-z']+", text)       
    found = set()                              
    for n in [2, 3, 4]:                            
        for i in range(len(words) - n + 1):
            phrase = words[i:i+n]
            if any(w in anchors for w in phrase): 
                found.add(" ".join(phrase))
    counts.update(found)                          


top = pd.DataFrame(counts.most_common(100), columns=["phrase", "reviews_with_it"])
top["percent_of_reviews"] = (top["reviews_with_it"] / len(dis) * 100).round(2)
if os.path.exists("lexicons/frequent_phrases.csv"):
    print("frequent_phrases.csv already exists.")
else:
    top.to_csv("lexicons/frequent_phrases.csv", index=False)
top.head(100)

frequent_phrases.csv already exists.


,phrase,reviews_with_it,percent_of_reviews
0,because i,631,3.75
1,because it,548,3.26
2,because of,327,1.94
3,because the,320,1.90
4,the reviews,274,1.63
...,...,...,...
95,was because,31,0.18
96,product because it,31,0.18
97,stars because it,30,0.18
98,understand why,29,0.17


In [ ]:
reasons = ["because", "hence", "as a result", "therefore", "thus", "due to", "the reason",
           "which is why", "that's why", "reason why", "only reason", "is why", "reason for"]

mentions = ["other reviews", "other reviewers", "read the reviews", "based on the reviews",
            "according to the reviews", "the reviews", "good reviews", "bad reviews",
            "great reviews", "positive reviews", "negative reviews", "rave reviews",
            "star reviews", "five star reviews", "everyone says", "everyone said",
            "people say", "people said", "others say", "others said", "as others have said",
            "like others said", "as others", "despite the reviews", "don't know why everyone",
            "not sure why everyone", "some people say", "some people said", "some people love",
            "some people loved", "other people say", "other people said", "other people love",
            "other people loved"]


if os.path.exists("lexicons/lexicon_reasons.txt") and os.path.exists("lexicons/lexicon_mentions.txt"):
    print("lexicon_reasons.txt and lexicon_mentions.txt already exist.")
else:
    with open("lexicons/lexicon_reasons.txt", "w") as f:
        f.write("\n".join(reasons))
    with open("lexicons/lexicon_mentions.txt", "w") as f:
        f.write("\n".join(mentions))

def make_pattern(phrases):
    phrases = sorted(phrases, key=len, reverse=True)
    return re.compile(r"\b(" + "|".join(re.escape(p) for p in phrases) + r")\b")

reasons_re = make_pattern(reasons)
mentions_re = make_pattern(mentions)


text = reviews["text_clean"].str.lower().str.replace("’", "'", regex=False)

reviews["reasons_count"] = text.str.count(reasons_re)
reviews["mentions_count"] = text.str.count(mentions_re)


reviews["reasons_pct"] = reviews["reasons_count"] / reviews["n_words"] * 100
reviews["mentions_pct"] = reviews["mentions_count"] / reviews["n_words"] * 100

reviews["mentions_any"] = reviews["mentions_count"] > 0


dis = reviews[reviews["dissent_neg"] & reviews["long_enough"]]
print("Number of bad reviews of good products:", len(dis))
print("Number of these reviews with at least one reason phrase:", (dis["reasons_count"] > 0).sum())
print("Number of these reviews that mention other reviewers:", dis["mentions_any"].sum())

lexicon_reasons.txt and lexicon_mentions.txt already exist.
Number of bad reviews of good products: 16816
Number of these reviews with at least one reason phrase: 2818
Number of these reviews that mention other reviewers: 795


In [ ]:

import numpy as np

dis = reviews[reviews["dissent_neg"] & reviews["long_enough"]].copy()
dis["review_id"] = dis.index   

dis["list_mentions"] = np.where(dis["mentions_any"], "yes", "no")
dis["list_reasons"]  = np.where(dis["reasons_count"] > 0, "yes", "no")

print(pd.crosstab(dis["list_mentions"], dis["list_reasons"]))

parts = []
for m in ["yes", "no"]:
    for r in ["yes", "no"]:
        group = dis[(dis["list_mentions"] == m) & (dis["list_reasons"] == r)]
        parts.append(group.sample(n=min(100, len(group)), random_state=42))
sample = pd.concat(parts)

sample = sample.sample(frac=1, random_state=42)

if os.path.exists("ai_labels/check_sample.csv") or os.path.exists("ai_labels/check_sample_answers.csv"):
    print("check_sample.csv and check_sample_answers.csv already exist - not overwritten.")
else:
    sample[["review_id", "text_clean"]].to_csv("ai_labels/check_sample.csv", index=False)
    sample[["review_id", "list_mentions", "list_reasons"]].to_csv("ai_labels/check_sample_answers.csv", index=False)

print(pd.crosstab(sample["list_mentions"], sample["list_reasons"]))
print("Sample size:", len(sample))

list_reasons      no   yes
list_mentions             
no             13453  2568
yes              545   250
check_sample.csv and check_sample_answers.csv already exist - not overwritten.
list_reasons    no  yes
list_mentions          
no             100  100
yes            100  100
Sample size: 400


Do not uncomment or run the next cell. It was run once to create `spot_check.csv`, which holds the 30 reviews that were labelled by hand. Running it again would replace the file with empty columns and erase the hand labels. It is kept only to show how the file was created.

In [ ]:
# sample_text = pd.read_csv("ai_labels/check_sample.csv")
# spot = sample_text.sample(30, random_state=7).copy()
# spot["my_mentions"] = ""
# spot["my_reasons"] = ""
# spot.to_csv("manual_checks/spot_check.csv", index=False)